# Notebook for the 1000 Runs Ensemble

In [32]:
import pandas as pd
import os
from utils.eda_utils import EDAUtils
import boto3

In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [35]:
os.makedirs(SSP_DIR_PATH, exist_ok=True)

In [36]:
edau = EDAUtils()

## Pull data from AWS S3


In [37]:
aws_config = edau.read_yaml(os.path.join(CONFIG_DIR_PATH, "aws_credentials_config.yaml"))
profile_name = aws_config["profile_name"]
bucket_name = aws_config["bucket_name"]
# Set your profile
session = boto3.Session(profile_name=profile_name)

# Create an S3 client or resource
s3 = session.resource('s3')

# Define folder prefix
prefix = 'transfers/sisepuede_run_2025-08-28t15;29;22.344855/'  # this is like the "folder" in S3

In [38]:
# Local destination
destination = os.path.join(SSP_DIR_PATH, prefix.strip('/'))
if os.path.exists(destination) and os.listdir(destination):
    print(f"Destination '{destination}' already exists and is not empty. Skipping download.")
else:
    os.makedirs(destination, exist_ok=True)
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=prefix):
        if obj.key.endswith('/'):  # skip directories
            continue
        target_path = os.path.join(destination, os.path.basename(obj.key))
        print(f"Downloading {obj.key} to {target_path}")
        bucket.download_file(obj.key, target_path)
        print(f"Downloaded: {obj.key}")

Destination '/Users/tony/Documents/sisepuede_modeling/ssp_louisiana/metamodel/data/ssp/transfers/sisepuede_run_2025-08-28t15;29;22.344855' already exists and is not empty. Skipping download.


In [39]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, prefix.strip('/'))

## Load and Process LHC Samples Dataframes

In [40]:
# Load lhc samples dfs
lhs_exogenous_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES.csv"))
lhs_levers_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_LEVER_EFFECTS.csv"))

In [48]:
# Check design ids
lhs_exogenous_df.head()

,region,design_id,future_id,47,48,49,50,51,52,53,54,55,56,57,58
0,louisiana,-1,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,0.681412,0.199758,0.969220
1,louisiana,-1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,0.000798,0.725034,0.902345
2,louisiana,-1,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,0.575067,0.208350,0.175365
3,louisiana,-1,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,0.506764,0.227383,0.113647
4,louisiana,-1,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,0.362234,0.947292,0.764492


In [49]:
lhs_exogenous_df.design_id.unique()

array([-1,  4])

In [50]:
lhs_levers_df.head()

,region,design_id,future_id,1,2,3,4,5,6,7,...,1753,1754,1759,1760,1762,1770,1772,1784,1785,1793
0,louisiana,-1,1,0.354682,0.681300,0.670551,0.371381,0.597378,0.390717,0.554668,...,0.406492,0.785543,0.249585,0.808708,0.824054,0.596933,0.085905,0.349450,0.718493,0.920739
1,louisiana,-1,2,0.023662,0.659807,0.066549,0.350365,0.536160,0.040042,0.101776,...,0.133208,0.725231,0.615082,0.234993,0.342072,0.975031,0.431114,0.700254,0.246668,0.536234
2,louisiana,-1,3,0.414641,0.092022,0.227137,0.020454,0.832906,0.293800,0.601890,...,0.320059,0.836571,0.386288,0.728487,0.069142,0.128675,0.699301,0.166701,0.354644,0.300748
3,louisiana,-1,4,0.307337,0.204967,0.050181,0.805452,0.558369,0.506111,0.974177,...,0.318834,0.118582,0.767219,0.762363,0.120502,0.838756,0.298101,0.885792,0.274482,0.827713
4,louisiana,-1,5,0.800355,0.458285,0.101171,0.738594,0.254548,0.034029,0.673997,...,0.954028,0.331731,0.708760,0.200289,0.889109,0.238305,0.473463,0.147280,0.491020,0.480479


In [51]:
lhs_levers_df.design_id.unique()

array([-1,  4])

In [57]:
# print shapes
print(lhs_exogenous_df.shape)
print(lhs_levers_df.shape)

(2000, 15)
(2000, 73)


In [52]:
lhs_df_merged = pd.merge(lhs_exogenous_df, lhs_levers_df, on=["region", "design_id", "future_id"], how="outer", suffixes=('_X', '_L'))
lhs_df_merged.head()

,region,design_id,future_id,47,48,49,50,51,52,53,...,1753,1754,1759,1760,1762,1770,1772,1784,1785,1793
0,louisiana,-1,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,...,0.406492,0.785543,0.249585,0.808708,0.824054,0.596933,0.085905,0.349450,0.718493,0.920739
1,louisiana,-1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,...,0.133208,0.725231,0.615082,0.234993,0.342072,0.975031,0.431114,0.700254,0.246668,0.536234
2,louisiana,-1,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,...,0.320059,0.836571,0.386288,0.728487,0.069142,0.128675,0.699301,0.166701,0.354644,0.300748
3,louisiana,-1,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,...,0.318834,0.118582,0.767219,0.762363,0.120502,0.838756,0.298101,0.885792,0.274482,0.827713
4,louisiana,-1,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,...,0.954028,0.331731,0.708760,0.200289,0.889109,0.238305,0.473463,0.147280,0.491020,0.480479


In [54]:
# Filter the lhs_df_merged to only include rows where design_id is 4
lhs_df_merged = lhs_df_merged[lhs_df_merged.design_id == 4]
lhs_df_merged.shape

(1000, 85)

In [55]:
lhs_df_merged.design_id.unique()

array([4])

In [56]:
# NOTE: check col names, there should be no duplicates
lhs_df_merged.columns

Index(['region', 'design_id', 'future_id', '47', '48', '49', '50', '51', '52',
       '53', '54', '55', '56', '57', '58', '1', '2', '3', '4', '5', '6', '7',
       '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19',
       '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31',
       '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43',
       '44', '45', '46', '1482', '1483', '1485', '1487', '1706', '1707',
       '1710', '1712', '1715', '1732', '1733', '1736', '1739', '1741', '1753',
       '1754', '1759', '1760', '1762', '1770', '1772', '1784', '1785', '1793'],
      dtype='object')

In [58]:
lhs_df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1000 to 1999
Data columns (total 85 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   region     1000 non-null   object 
 1   design_id  1000 non-null   int64  
 2   future_id  1000 non-null   int64  
 3   47         1000 non-null   float64
 4   48         1000 non-null   float64
 5   49         1000 non-null   float64
 6   50         1000 non-null   float64
 7   51         1000 non-null   float64
 8   52         1000 non-null   float64
 9   53         1000 non-null   float64
 10  54         1000 non-null   float64
 11  55         1000 non-null   float64
 12  56         1000 non-null   float64
 13  57         1000 non-null   float64
 14  58         1000 non-null   float64
 15  1          1000 non-null   float64
 16  2          1000 non-null   float64
 17  3          1000 non-null   float64
 18  4          1000 non-null   float64
 19  5          1000 non-null   float64
 20  6         

## Load SISEPUEDE WIDE_INPUTS_OUTPUTS

In [59]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "sisepuede_results_IDE_2025-08-28t15;29;22.344855_only_6004_and_baseline.csv"))
wide_inputs_outputs_df

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
0,332332,louisiana,7,0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,1.119173e+06,...,117.042622,2.762559,-0.077001,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402
1,332332,louisiana,8,0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,1.114548e+06,...,160.204394,2.773548,-0.104854,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568
2,332332,louisiana,9,0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,1.109930e+06,...,116.085554,2.786645,-0.132605,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706
3,332332,louisiana,10,0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,1.105320e+06,...,163.932900,2.801782,-0.160253,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336
4,332332,louisiana,11,0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,1.100718e+06,...,163.431885,2.818877,-0.187795,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28763,403402,louisiana,31,0,362765.501037,62973.310285,84.270736,82487.288241,7362.303302,1.185080e+06,...,169.371247,2.075427,-1.990093,0.193981,1.441077,1.211796,1.706858,36.976660,0.504737,5.271994
28764,403402,louisiana,32,0,365228.715699,63193.456636,85.153817,83260.287378,7453.568466,1.195508e+06,...,169.256140,2.031856,-2.010800,0.192032,1.456824,1.041882,1.778988,36.260617,0.504923,5.373060
28765,403402,louisiana,33,0,367719.988263,63434.430550,86.031953,84023.290062,7543.331453,1.205869e+06,...,169.154549,1.986896,-2.048092,0.189841,1.471911,0.871422,1.846095,35.524647,0.504966,5.474498
28766,403402,louisiana,34,0,370220.245668,63693.168311,86.899932,84771.705654,7631.119506,1.216095e+06,...,169.064989,1.940356,-2.098197,0.187394,1.486149,0.700168,1.891870,34.765612,0.504860,5.576305


## Load Costs-Benefits Data

In [18]:
cb_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "wide_cb_data_lhc_2025-08-10t10;29;30.545790.csv"))
cb_df.head()

,primary_id,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,...,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,354355,1,PFLO:ALL_LA_ACTIONS,2022.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,354355,1,PFLO:ALL_LA_ACTIONS,2023.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,354355,1,PFLO:ALL_LA_ACTIONS,2024.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,354355,1,PFLO:ALL_LA_ACTIONS,2025.0,0.060282,0.132495,0.021720,4.143264e-14,-3.977853e-13,0.000014,...,0.018795,-0.053377,-1.193712e-17,-0.131033,0.210159,7536.242379,37.153191,0.619617,0.207821,-0.025037
4,354355,1,PFLO:ALL_LA_ACTIONS,2026.0,0.073026,0.159321,0.043711,-1.172087e-14,4.620699e-13,0.000017,...,0.038597,-0.057632,1.875833e-17,-0.156095,0.252749,9164.677971,45.045727,-0.196941,0.192236,-0.030054


In [19]:
cb_df.tail()

,primary_id,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,...,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
28821,355354,1000,PFLO:ALL_LA_ACTIONS,2046.0,-1.423236,-1.098143,0.440375,-0.610346,-0.055704,0.005134,...,0.404066,0.015166,-0.001725,3.338973,-1.600469,-51911.920163,-324.093158,-5.409679,0.956401,-0.016539
28822,355354,1000,PFLO:ALL_LA_ACTIONS,2047.0,-1.461855,-1.133005,0.460031,-0.640778,-0.061686,0.005436,...,0.423545,0.015450,-0.001842,3.558048,-1.649368,-55304.937750,-344.996680,-3.833681,0.986141,-0.017252
28823,355354,1000,PFLO:ALL_LA_ACTIONS,2048.0,-1.500225,-1.166798,0.479711,-0.670230,-0.067975,0.005741,...,0.443083,0.015812,-0.001959,3.781217,-1.696533,-58836.758704,-366.532141,-21.918525,1.015314,-0.017971
28824,355354,1000,PFLO:ALL_LA_ACTIONS,2049.0,-1.538374,-1.199517,0.499414,-0.698952,-0.074553,0.006049,...,0.462685,0.016260,-0.002076,4.008499,-1.741936,-62510.293177,-388.696951,-4.158967,1.043825,-0.018697
28825,355354,1000,PFLO:ALL_LA_ACTIONS,2050.0,-1.576095,-1.231121,0.519141,-0.727129,-0.081400,0.006362,...,0.482357,0.016801,-0.002193,4.239890,-1.785494,-66302.419876,-411.476185,-3.434625,1.071532,-0.019430


## Data Cleaning

### SISEPUEDE Emission data

In [20]:
# Get the subsector total variables
subsector_total_vars = [c for c in wide_inputs_outputs_df.columns if "emission_co2e_subsector_total" in c]

In [21]:
# Filter to only subsector total columns and primary_id, time_period
la_emissions_df = wide_inputs_outputs_df[["primary_id", "time_period"] + subsector_total_vars]
la_emissions_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu
0,354660,7,2.923974,-36.545088,2.050000,0.230319,1.539811,1.233181,3.138402,0.447204,0.0,30.469259,13.094104,117.042622,4.491483,45.130223,2.629950
1,354660,8,2.863991,-39.756859,2.039113,0.229380,1.516608,1.227027,3.190377,0.453932,0.0,38.785521,15.448650,114.205980,4.519073,46.068411,2.641921
2,354660,9,2.911902,-42.274952,2.028288,0.228514,1.493765,1.210373,3.241353,0.460975,0.0,35.280608,14.868080,114.046420,4.549158,47.114101,2.656241
3,354660,10,2.899808,-44.341002,2.017523,0.227722,1.471286,1.181755,3.293836,0.468262,0.0,47.213230,17.172010,112.604235,4.581895,48.249651,2.672838
4,354660,11,2.887734,-46.106391,2.006818,0.226980,1.449157,1.140940,3.347158,0.475743,0.0,47.401636,17.516973,111.117115,4.617352,49.461277,2.691630


In [22]:
la_emissions_df.tail()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu
28850,355313,31,3.548737,-70.869178,2.277054,0.167744,1.373834,2.905346,5.297445,0.552432,-1.199164,46.662677,14.159895,62.516491,3.070915,51.396540,2.243853
28851,355313,32,3.582179,-72.277471,2.213204,0.161924,1.341855,2.913548,5.406589,0.554898,-1.262278,46.851672,13.945969,60.984318,2.970735,51.503706,2.218678
28852,355313,33,3.613989,-73.719628,2.157949,0.156007,1.308322,2.919315,5.516952,0.557270,-1.325391,47.059094,13.732943,59.489538,2.867692,51.602941,2.192627
28853,355313,34,3.644501,-75.192293,2.109828,0.150024,1.273528,2.904608,5.628525,0.559544,-1.388505,47.284998,13.520595,58.034030,2.761373,51.691903,2.165547
28854,355313,35,3.674001,-76.693931,2.099561,0.143996,1.237701,2.890354,5.741263,0.561709,-1.451619,47.510166,13.303400,57.496195,2.651367,51.767368,2.137253


### CB Data

In [23]:
# Make all column names lowercase
cb_df.columns = [c.lower() for c in cb_df.columns]

# Filter to only important cb columns
cb_df = cb_df[["primary_id",
               "future_id",
               "year",
               "technical_cost"]]

cb_df.head()

,primary_id,future_id,year,technical_cost
0,354355,1,2022.0,0.000000
1,354355,1,2023.0,0.000000
2,354355,1,2024.0,0.000000
3,354355,1,2025.0,0.619617
4,354355,1,2026.0,-0.196941


In [24]:
cb_df.tail()

,primary_id,future_id,year,technical_cost
28821,355354,1000,2046.0,-5.409679
28822,355354,1000,2047.0,-3.833681
28823,355354,1000,2048.0,-21.918525
28824,355354,1000,2049.0,-4.158967
28825,355354,1000,2050.0,-3.434625


In [25]:
cb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28826 entries, 0 to 28825
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   primary_id      28826 non-null  int64  
 1   future_id       28826 non-null  int64  
 2   year            28826 non-null  float64
 3   technical_cost  28826 non-null  float64
dtypes: float64(2), int64(2)
memory usage: 900.9 KB


## LHS Data

In [26]:
lhs_df_merged = lhs_df_merged.drop(columns=["design_id", "region"])
lhs_df_merged.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1739,1740,1745,1746,1748,1756,1758,1770,1771,1779
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.367236,0.680655,0.917957,0.134558,0.193991,0.644576,0.214517,0.955397,0.733289,0.116270
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.840756,0.705425,0.984197,0.409837,0.514426,0.486411,0.278740,0.680327,0.626129,0.392241
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.691967,0.816298,0.709269,0.917388,0.682384,0.331853,0.193970,0.968824,0.467563,0.402772
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.748838,0.617573,0.971575,0.477516,0.762533,0.267855,0.674344,0.246455,0.464470,0.968928
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.942745,0.207604,0.936352,0.371713,0.712287,0.135855,0.111477,0.455590,0.378535,0.267191


## Transform time series format into single-row format

### SISEPUEDE Emission data

In [27]:
# Sum all the subsector emission columns across axis=1
la_emission_total_df = la_emissions_df.copy()
la_emission_total_df["emission_total"] = la_emission_total_df[subsector_total_vars].sum(axis=1)
la_emission_total_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu,emission_total
0,354660,7,2.923974,-36.545088,2.050000,0.230319,1.539811,1.233181,3.138402,0.447204,0.0,30.469259,13.094104,117.042622,4.491483,45.130223,2.629950,187.875442
1,354660,8,2.863991,-39.756859,2.039113,0.229380,1.516608,1.227027,3.190377,0.453932,0.0,38.785521,15.448650,114.205980,4.519073,46.068411,2.641921,193.433125
2,354660,9,2.911902,-42.274952,2.028288,0.228514,1.493765,1.210373,3.241353,0.460975,0.0,35.280608,14.868080,114.046420,4.549158,47.114101,2.656241,187.814826
3,354660,10,2.899808,-44.341002,2.017523,0.227722,1.471286,1.181755,3.293836,0.468262,0.0,47.213230,17.172010,112.604235,4.581895,48.249651,2.672838,199.713047
4,354660,11,2.887734,-46.106391,2.006818,0.226980,1.449157,1.140940,3.347158,0.475743,0.0,47.401636,17.516973,111.117115,4.617352,49.461277,2.691630,198.234123


In [28]:
# Keep only the primary_id, time_period, and emission_total columns
la_emission_total_df = la_emission_total_df[["primary_id", "time_period", "emission_total"]]
la_emission_total_df.head()

,primary_id,time_period,emission_total
0,354660,7,187.875442
1,354660,8,193.433125
2,354660,9,187.814826
3,354660,10,199.713047
4,354660,11,198.234123


In [29]:
la_emission_total_df.tail()

,primary_id,time_period,emission_total
28850,355313,31,124.104622
28851,355313,32,121.109526
28852,355313,33,118.129621
28853,355313,34,115.148206
28854,355313,35,113.068783


### Emission data sum

In [30]:
# aggregate data by primary_id summing the emissions
la_emission_df_sum_agg = la_emission_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_emission_df_sum_agg = la_emission_df_sum_agg.drop(columns=["time_period"])
la_emission_df_sum_agg.head()

,primary_id,emission_total
0,354354,1700.012557
1,354355,3553.270010
2,354356,3028.607700
3,354357,4127.536839
4,354358,2668.240547


### Emission data mean

In [31]:
# Filter out rows with time_period < 31
la_filtered_emission_total_df = la_emission_total_df[la_emission_total_df["time_period"] >= 31]
la_filtered_emission_total_df = la_filtered_emission_total_df.reset_index(drop=True)
la_filtered_emission_total_df.head(7)

,primary_id,time_period,emission_total
0,354660,31,53.155458
1,354660,32,49.182152
2,354660,33,45.371484
3,354660,34,41.661921
4,354660,35,38.180330
5,354533,31,24.009496
6,354533,32,19.425322


In [32]:
# aggregate data by primary_id by summing the emissions
la_emission_df_mean_agg = la_filtered_emission_total_df.groupby(["primary_id"]).mean().reset_index()

# Rename emission_total to emission_avg_last_five_years
la_emission_df_mean_agg.rename(columns={"emission_total": "emission_avg_last_five_years"}, inplace=True)
la_emission_df_mean_agg

,primary_id,time_period,emission_avg_last_five_years
0,354354,33.0,-54.039699
1,354355,33.0,59.720047
2,354356,33.0,30.972144
3,354357,33.0,92.551550
4,354358,33.0,11.015231
...,...,...,...
990,355350,33.0,23.599187
991,355351,33.0,60.727431
992,355352,33.0,51.497226
993,355353,33.0,62.552429


In [33]:
# Drop year column as it is no longer needed
la_emission_df_mean_agg = la_emission_df_mean_agg.drop(columns=["time_period"])
la_emission_df_mean_agg.head()

,primary_id,emission_avg_last_five_years
0,354354,-54.039699
1,354355,59.720047
2,354356,30.972144
3,354357,92.551550
4,354358,11.015231


### Combining emission agg into one df

In [34]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)
print("la_emission_df_sum_agg shape:", la_emission_df_sum_agg.shape)

la_emission_df_mean_agg shape: (995, 2)
la_emission_df_sum_agg shape: (995, 2)


In [35]:
la_emissions_df_merged = la_emission_df_mean_agg.merge(la_emission_df_sum_agg, on="primary_id", how="inner")
la_emissions_df_merged.head()

,primary_id,emission_avg_last_five_years,emission_total
0,354354,-54.039699,1700.012557
1,354355,59.720047,3553.270010
2,354356,30.972144,3028.607700
3,354357,92.551550,4127.536839
4,354358,11.015231,2668.240547


In [36]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)

la_emission_df_mean_agg shape: (995, 2)


### CB data

In [37]:
# aggregate data by primary_id and region by summing the technical cost
cb_df_agg = cb_df.groupby(["primary_id", "future_id"]).sum().reset_index()
cb_df_agg

,primary_id,future_id,year,technical_cost
0,354355,1,59044.0,-42.336438
1,354356,2,59044.0,-21.656242
2,354357,3,59044.0,-40.157702
3,354358,4,59044.0,-53.155978
4,354359,5,59044.0,-77.798140
...,...,...,...,...
989,355350,996,59044.0,-51.800498
990,355351,997,59044.0,-23.938700
991,355352,998,59044.0,-129.619974
992,355353,999,59044.0,-132.245620


In [38]:
# Drop year column as it is no longer needed
cb_df_agg = cb_df_agg.drop(columns=["year"], errors='ignore')
cb_df_agg.head()

,primary_id,future_id,technical_cost
0,354355,1,-42.336438
1,354356,2,-21.656242
2,354357,3,-40.157702
3,354358,4,-53.155978
4,354359,5,-77.798140


In [39]:
cb_df_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 994 entries, 0 to 993
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   primary_id      994 non-null    int64  
 1   future_id       994 non-null    int64  
 2   technical_cost  994 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 23.4 KB


## Merge emissions and cb data with lhs samples

In [40]:
attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
0,354354,4,6004,0
1,354355,4,6004,1
2,354356,4,6004,2
3,354357,4,6004,3
4,354358,4,6004,4
...,...,...,...,...
996,355350,4,6004,996
997,355351,4,6004,997
998,355352,4,6004,998
999,355353,4,6004,999


In [41]:
# check for duplicates primary_id
duplicates = attr_primary_df[attr_primary_df.duplicated(subset=["primary_id"], keep=False)]
if not duplicates.empty:
    print("Duplicated primary_id found:")
    print(duplicates)
else:
    print("No duplicated primary_id found.")

No duplicated primary_id found.


In [42]:
la_emission_df_w_future_id = la_emissions_df_merged.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_emission_df_w_future_id = la_emission_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_emission_df_w_future_id

,primary_id,emission_avg_last_five_years,emission_total,future_id
0,354354,-54.039699,1700.012557,0
1,354355,59.720047,3553.270010,1
2,354356,30.972144,3028.607700,2
3,354357,92.551550,4127.536839,3
4,354358,11.015231,2668.240547,4
...,...,...,...,...
990,355350,23.599187,2958.134476,996
991,355351,60.727431,3526.674139,997
992,355352,51.497226,3434.786875,998
993,355353,62.552429,3695.679453,999


In [43]:
lhs_df_merged

,future_id,47,48,49,50,51,52,53,54,55,...,1739,1740,1745,1746,1748,1756,1758,1770,1771,1779
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.367236,0.680655,0.917957,0.134558,0.193991,0.644576,0.214517,0.955397,0.733289,0.116270
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.840756,0.705425,0.984197,0.409837,0.514426,0.486411,0.278740,0.680327,0.626129,0.392241
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.691967,0.816298,0.709269,0.917388,0.682384,0.331853,0.193970,0.968824,0.467563,0.402772
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.748838,0.617573,0.971575,0.477516,0.762533,0.267855,0.674344,0.246455,0.464470,0.968928
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.942745,0.207604,0.936352,0.371713,0.712287,0.135855,0.111477,0.455590,0.378535,0.267191
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,996,0.736701,0.546430,0.739303,0.054172,0.451181,0.875108,0.854104,0.601669,0.364533,...,0.923277,0.404664,0.807382,0.023082,0.141101,0.404654,0.502693,0.552017,0.582446,0.800677
1996,997,0.484244,0.232193,0.411141,0.677399,0.536223,0.275901,0.268436,0.277561,0.699502,...,0.121316,0.519401,0.017957,0.448007,0.290108,0.492823,0.028926,0.811945,0.602409,0.508916
1997,998,0.594076,0.453653,0.999049,0.260340,0.276599,0.669046,0.335968,0.251319,0.205155,...,0.377285,0.500519,0.213150,0.061383,0.371412,0.278274,0.968587,0.789580,0.138852,0.497848
1998,999,0.991327,0.185911,0.093308,0.002969,0.027930,0.843634,0.806122,0.438407,0.116038,...,0.714758,0.021488,0.426470,0.839947,0.467256,0.773997,0.174197,0.194015,0.439769,0.640290


In [44]:
lhs_emissions_merged_df = pd.merge(lhs_df_merged, la_emission_df_w_future_id, on="future_id", how="inner")
lhs_emissions_merged_df.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1746,1748,1756,1758,1770,1771,1779,primary_id,emission_avg_last_five_years,emission_total
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.134558,0.193991,0.644576,0.214517,0.955397,0.733289,0.116270,354355,59.720047,3553.270010
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.409837,0.514426,0.486411,0.278740,0.680327,0.626129,0.392241,354356,30.972144,3028.607700
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.917388,0.682384,0.331853,0.193970,0.968824,0.467563,0.402772,354357,92.551550,4127.536839
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.477516,0.762533,0.267855,0.674344,0.246455,0.464470,0.968928,354358,11.015231,2668.240547
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.371713,0.712287,0.135855,0.111477,0.455590,0.378535,0.267191,354359,41.876734,3343.173022


In [45]:
complete_merged_df = pd.merge(lhs_emissions_merged_df, cb_df_agg, on=["future_id", "primary_id"], how="inner")
complete_merged_df.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1748,1756,1758,1770,1771,1779,primary_id,emission_avg_last_five_years,emission_total,technical_cost
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.193991,0.644576,0.214517,0.955397,0.733289,0.116270,354355,59.720047,3553.270010,-42.336438
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.514426,0.486411,0.278740,0.680327,0.626129,0.392241,354356,30.972144,3028.607700,-21.656242
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.682384,0.331853,0.193970,0.968824,0.467563,0.402772,354357,92.551550,4127.536839,-40.157702
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.762533,0.267855,0.674344,0.246455,0.464470,0.968928,354358,11.015231,2668.240547,-53.155978
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.712287,0.135855,0.111477,0.455590,0.378535,0.267191,354359,41.876734,3343.173022,-77.798140


In [46]:
# rearrange columns to have future_id and primary_id at the front
cols_order = ["future_id", "primary_id"] + [col for col in complete_merged_df.columns if col not in ["future_id", "primary_id"]]
complete_merged_df = complete_merged_df[cols_order]
complete_merged_df

,future_id,primary_id,47,48,49,50,51,52,53,54,...,1746,1748,1756,1758,1770,1771,1779,emission_avg_last_five_years,emission_total,technical_cost
0,1,354355,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,...,0.134558,0.193991,0.644576,0.214517,0.955397,0.733289,0.116270,59.720047,3553.270010,-42.336438
1,2,354356,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,...,0.409837,0.514426,0.486411,0.278740,0.680327,0.626129,0.392241,30.972144,3028.607700,-21.656242
2,3,354357,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,...,0.917388,0.682384,0.331853,0.193970,0.968824,0.467563,0.402772,92.551550,4127.536839,-40.157702
3,4,354358,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,...,0.477516,0.762533,0.267855,0.674344,0.246455,0.464470,0.968928,11.015231,2668.240547,-53.155978
4,5,354359,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,...,0.371713,0.712287,0.135855,0.111477,0.455590,0.378535,0.267191,41.876734,3343.173022,-77.798140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,996,355350,0.736701,0.546430,0.739303,0.054172,0.451181,0.875108,0.854104,0.601669,...,0.023082,0.141101,0.404654,0.502693,0.552017,0.582446,0.800677,23.599187,2958.134476,-51.800498
1984,997,355351,0.484244,0.232193,0.411141,0.677399,0.536223,0.275901,0.268436,0.277561,...,0.448007,0.290108,0.492823,0.028926,0.811945,0.602409,0.508916,60.727431,3526.674139,-23.938700
1985,998,355352,0.594076,0.453653,0.999049,0.260340,0.276599,0.669046,0.335968,0.251319,...,0.061383,0.371412,0.278274,0.968587,0.789580,0.138852,0.497848,51.497226,3434.786875,-129.619974
1986,999,355353,0.991327,0.185911,0.093308,0.002969,0.027930,0.843634,0.806122,0.438407,...,0.839947,0.467256,0.773997,0.174197,0.194015,0.439769,0.640290,62.552429,3695.679453,-132.245620


## Filter out irrelevant lhs groups

In [47]:
var_traj_X_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_X.csv"))
var_traj_L_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_L.csv"))

In [48]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
53,elasticity_ippu_wood_production_to_gdp,57
54,elasticity_ippu_product_use_lubricants_product...,58
55,elasticity_ippu_product_use_ods_other_product_...,58
56,elasticity_ippu_product_use_ods_refrigeration_...,58
57,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [49]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
470,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,46
471,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,46
472,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,46
473,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,46
474,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,46


In [50]:
var_traj_groups_X = var_traj_X_df["variable_trajectory_group"].unique()
var_traj_groups_L = var_traj_L_df["variable_trajectory_group"].unique()
print("Variable trajectory groups X:", var_traj_groups_X)
print("Variable trajectory groups L:", var_traj_groups_L)

Variable trajectory groups X: [47 48 49 50 51 52 53 54 55 56 57 58]
Variable trajectory groups L: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46]


In [51]:
# join the var_traj_groups
var_traj_groups_all = var_traj_groups_X.tolist() + var_traj_groups_L.tolist()
var_traj_groups_all = list(set(var_traj_groups_all))  # remove duplicates
print("All variable trajectory groups:", var_traj_groups_all)

All variable trajectory groups: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]


In [52]:
# Convert the variable_trajectory_group column to list of strings
relevant_lhs_cols = [str(col) for col in var_traj_groups_all]

In [53]:
df_cols = complete_merged_df.columns.tolist()

# Filter the relevant_lhs_cols to only include those that are in df_cols
relevant_lhs_cols = [col for col in relevant_lhs_cols if col in df_cols]

In [54]:
# filter complete_merged_df to keep only relevant columns
cols_to_keep = ["future_id", "primary_id"] + list(relevant_lhs_cols) + ["emission_avg_last_five_years", "emission_total", "technical_cost"]
merged_df_filtered = complete_merged_df[cols_to_keep]

In [55]:
print("Original merged DataFrame shape:", complete_merged_df.shape)
print("Filtered merged DataFrame shape:", merged_df_filtered.shape)

Original merged DataFrame shape: (1988, 83)
Filtered merged DataFrame shape: (1988, 63)


In [56]:
print("Filtered merged DataFrame fields:", merged_df_filtered.columns.tolist())
print("Relevant LHS columns:", relevant_lhs_cols)

Filtered merged DataFrame fields: ['future_id', 'primary_id', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', 'emission_avg_last_five_years', 'emission_total', 'technical_cost']
Relevant LHS columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58']


In [57]:
merged_df_filtered.head()

,future_id,primary_id,1,2,3,4,5,6,7,8,...,52,53,54,55,56,57,58,emission_avg_last_five_years,emission_total,technical_cost
0,1,354355,0.709094,0.362421,0.352636,0.246076,0.230763,0.231633,0.357253,0.416035,...,0.065254,0.452244,0.395609,0.190891,0.681412,0.199758,0.969220,59.720047,3553.270010,-42.336438
1,2,354356,0.355127,0.388826,0.495092,0.282101,0.198981,0.620631,0.428909,0.192468,...,0.017979,0.912414,0.894009,0.046745,0.000798,0.725034,0.902345,30.972144,3028.607700,-21.656242
2,3,354357,0.970473,0.436334,0.243523,0.227546,0.467160,0.102438,0.605842,0.087344,...,0.416929,0.832413,0.406103,0.929365,0.575067,0.208350,0.175365,92.551550,4127.536839,-40.157702
3,4,354358,0.605323,0.210722,0.661197,0.823838,0.003365,0.990644,0.114855,0.215559,...,0.778987,0.995629,0.994153,0.946893,0.506764,0.227383,0.113647,11.015231,2668.240547,-53.155978
4,5,354359,0.247035,0.704965,0.523551,0.535350,0.345173,0.482609,0.894310,0.862094,...,0.918567,0.183357,0.397512,0.328322,0.362234,0.947292,0.764492,41.876734,3343.173022,-77.798140


## Add variable names to lhs columns

In [58]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
470,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,46
471,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,46
472,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,46
473,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,46
474,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,46


In [59]:
var_traj_L_df = var_traj_L_df[["variable_field", "variable_trajectory_group"]]
var_traj_L_df = var_traj_L_df.rename(columns={"variable_field": "variable"})
var_traj_L_df.head()

,variable,variable_trajectory_group
0,ef_agrc_anaerobicdom_rice_kg_ch4_ha,1
1,frac_agrc_agriculture_production_lost,2
2,frac_agrc_crop_residues_burned,3
3,frac_agrc_crop_residues_removed,3
4,frac_agrc_no_till_cereals,3


In [60]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
53,elasticity_ippu_wood_production_to_gdp,57
54,elasticity_ippu_product_use_lubricants_product...,58
55,elasticity_ippu_product_use_ods_other_product_...,58
56,elasticity_ippu_product_use_ods_refrigeration_...,58
57,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [61]:
var_traj_all_df = pd.concat([var_traj_X_df, var_traj_L_df], ignore_index=True)
var_traj_all_df

,variable,variable_trajectory_group
0,cost_enfu_fuel_coal_usd_per_tonne,47
1,cost_enfu_fuel_coke_usd_per_tonne,47
2,cost_enfu_fuel_hydrocarbon_gas_liquids_usd_per...,47
3,cost_enfu_fuel_natural_gas_usd_per_mmbtu,47
4,cost_enfu_fuel_crude_usd_per_m3,47
...,...,...
528,frac_waso_recycled_paper,46
529,frac_waso_recycled_plastic,46
530,frac_waso_recycled_rubber_leather,46
531,frac_waso_recycled_textiles,46


In [62]:
# drop duplicates if any
print("Before dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)
var_traj_all_df = var_traj_all_df.drop_duplicates(subset=["variable", "variable_trajectory_group"])
print("After dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)

Before dropping duplicates, var_traj_all_df shape: (533, 2)
After dropping duplicates, var_traj_all_df shape: (520, 2)


In [63]:
# check if there are any duplicated variable names
duplicated_vars = var_traj_all_df["variable"].duplicated().any()
if duplicated_vars:
    print("There are duplicated variable names in var_traj_all_df.")
else:
    print("No duplicated variable names in var_traj_all_df.")

No duplicated variable names in var_traj_all_df.


In [64]:
# Filter var_traj_all_df by sample_group in relevant_lhs_cols
relevant_lhs_cols = [int(col) for col in relevant_lhs_cols]
var_traj_all_df = var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin(relevant_lhs_cols)]
var_traj_all_df = var_traj_all_df.sort_values(by="variable_trajectory_group", ascending=True)
print("After filtering by relevant_lhs_cols, var_traj_all_df shape:", var_traj_all_df.shape)

After filtering by relevant_lhs_cols, var_traj_all_df shape: (520, 2)


In [65]:
def process_variable_prefix(df):
    result = []
    for group, group_df in df.groupby('variable_trajectory_group'):
        variables = group_df['variable'].tolist()
        if len(variables) == 1:
            prefix = variables[0]
        else:
            prefix = os.path.commonprefix(variables)
            # Clean trailing underscores
            prefix = prefix.rstrip('_')
            
        prefix = f"group_{group}_{prefix}"
        result.append({'variable_trajectory_group': group, 'variable_prefix': prefix})
    return pd.DataFrame(result)

prefix_df = process_variable_prefix(var_traj_all_df)
prefix_df

,variable_trajectory_group,variable_prefix
0,1,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha
1,2,group_2_frac_agrc_agriculture_production_lost
2,3,group_3_frac_agrc
3,4,group_4_qty_ccsq_mt_co2_captured_sequestered_b...
4,5,group_5_frac_enfu_transmission_loss_fuel_elect...
5,6,group_6_nemomod_en
6,7,group_7_nemomod_entc_frac_min_share_production...
7,8,group_8_frac_fgtv_reduction_in_fugitive_leaks
8,9,group_9_frac_fgtv_drained_and_waste_ch4_flared...
9,10,group_10_efficfactor_enfu_industrial_energy_fuel


In [66]:
# Check for duplicates in variable_trajectory_group and variable_prefix
dups = prefix_df.duplicated(subset=["variable_trajectory_group", "variable_prefix"], keep=False)
if dups.any():
    print("Duplicated variable_trajectory_group and variable_prefix found:")
    print(prefix_df[dups])
else:
    print("No duplicated variable_trajectory_group and variable_prefix found.")

# Check for duplicates in variable_trajectory_group
dups_group = prefix_df.duplicated(subset=["variable_trajectory_group"], keep=False)
if dups_group.any():
    print("Duplicated variable_trajectory_group found:")
    print(prefix_df[dups_group])
else:
    print("No duplicated variable_trajectory_group found.")

# Check for duplicates in variable_prefix
dups_prefix = prefix_df.duplicated(subset=["variable_prefix"], keep=False)
if dups_prefix.any():
    print("Duplicated variable_prefix found:")
    print(prefix_df[dups_prefix])
else:
    print("No duplicated variable_prefix found.")

No duplicated variable_trajectory_group and variable_prefix found.
No duplicated variable_trajectory_group found.
No duplicated variable_prefix found.


In [67]:
# var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin([3, 13, 40])]

In [68]:
# prefix_df.loc[prefix_df["sample_group"] == 13, "variable_prefix"] = "group_13_frac_gnrl_eating_red_meats+"
# prefix_df.loc[prefix_df["sample_group"] == 40, "variable_prefix"] = "group_40_pij_lndu_grasslands+"

# prefix_df = prefix_df.sort_values(by="variable_prefix", ascending=True)
# prefix_df

In [69]:
# Let's use the prefix_df to rename the columns in merged_df_filtered
def rename_columns_with_prefix(merged_df, prefix_df):
    df = merged_df.copy()
    # Create a mapping from str(group) to prefix
    group_to_prefix = {str(row['variable_trajectory_group']): row['variable_prefix'] for _, row in prefix_df.iterrows()}
    # Only rename columns that match a group
    rename_dict = {col: group_to_prefix[col] for col in df.columns if col in group_to_prefix}
    df = df.rename(columns=rename_dict)
    return df

merged_df_filtered_w_prefix = rename_columns_with_prefix(merged_df_filtered, prefix_df)

In [70]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_frac_enfu_transmission_loss_fuel_electricity,group_6_nemomod_en,group_7_nemomod_entc_frac_min_share_production_fp_hydrogen_electrolysis,group_8_frac_fgtv_reduction_in_fugitive_leaks,...,group_52_gdp_mmm_usd,group_53_population_gnrl,group_54_elasticity_scoe_enerdem_per,group_55_elasticity_trde_mtkm_to_gdp_freight,group_56_elasticity_trde_pkm_to_gdppc,group_57_elasticity_ippu,group_58_elasticity_ippu_product_use,emission_avg_last_five_years,emission_total,technical_cost
0,1,354355,0.709094,0.362421,0.352636,0.246076,0.230763,0.231633,0.357253,0.416035,...,0.065254,0.452244,0.395609,0.190891,0.681412,0.199758,0.969220,59.720047,3553.270010,-42.336438
1,2,354356,0.355127,0.388826,0.495092,0.282101,0.198981,0.620631,0.428909,0.192468,...,0.017979,0.912414,0.894009,0.046745,0.000798,0.725034,0.902345,30.972144,3028.607700,-21.656242
2,3,354357,0.970473,0.436334,0.243523,0.227546,0.467160,0.102438,0.605842,0.087344,...,0.416929,0.832413,0.406103,0.929365,0.575067,0.208350,0.175365,92.551550,4127.536839,-40.157702
3,4,354358,0.605323,0.210722,0.661197,0.823838,0.003365,0.990644,0.114855,0.215559,...,0.778987,0.995629,0.994153,0.946893,0.506764,0.227383,0.113647,11.015231,2668.240547,-53.155978
4,5,354359,0.247035,0.704965,0.523551,0.535350,0.345173,0.482609,0.894310,0.862094,...,0.918567,0.183357,0.397512,0.328322,0.362234,0.947292,0.764492,41.876734,3343.173022,-77.798140
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,996,355350,0.758811,0.639557,0.801335,0.366066,0.541587,0.853178,0.846512,0.184772,...,0.875108,0.854104,0.601669,0.364533,0.879831,0.204425,0.508420,23.599187,2958.134476,-51.800498
1984,997,355351,0.038856,0.120206,0.256071,0.181417,0.507199,0.590424,0.975452,0.482633,...,0.275901,0.268436,0.277561,0.699502,0.579293,0.826772,0.510035,60.727431,3526.674139,-23.938700
1985,998,355352,0.229752,0.060025,0.750683,0.297394,0.263480,0.148370,0.072600,0.332356,...,0.669046,0.335968,0.251319,0.205155,0.278802,0.359322,0.344483,51.497226,3434.786875,-129.619974
1986,999,355353,0.375548,0.122781,0.293197,0.977969,0.029632,0.088888,0.712697,0.304010,...,0.843634,0.806122,0.438407,0.116038,0.222832,0.415388,0.434470,62.552429,3695.679453,-132.245620


In [71]:
merged_df_filtered_w_prefix.shape

(1988, 63)

In [72]:
# check for duplicated column names
duplicated_cols = merged_df_filtered_w_prefix.columns[merged_df_filtered_w_prefix.columns.duplicated()].tolist()
if duplicated_cols:
    print("Duplicated column names found:", duplicated_cols)
else:
    print("No duplicated column names found.")

No duplicated column names found.


## Finally we save the processed data as training data

In [73]:
#save the merged DataFrame to a CSV file
merged_df_filtered_w_prefix.to_csv(os.path.join(TRAINING_DIR_PATH, "training_data_v4.2.csv"), index=False)